In [ ]:

!pip install -q -U bitsandbytes transformers peft accelerate datasets scipy einops evaluate trl rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 143.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 51.3 MB/s eta 0:00:00


In [ ]:
# mount to drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy
from sklearn.model_selection import train_test_split

In [ ]:
project_root = '/content/drive/MyDrive/Intelligent-Agents-Project/code'
data_root = project_root + '/data/drugs.csv'

dataset_dict = load_dataset("csv", data_files=data_root)
split_dataset = dataset_dict["train"].train_test_split(test_size=0.1, seed=42)

train_data = split_dataset["train"]
val_data = split_dataset["test"]


In [ ]:
def format_instruction(example):
    return {
        "text": f'''<s>[INST]
        You are Drug Expert. You'll have two drugs and you have to determine whether they interact or not.
        Drug 1: {example['Drug_A']}
        Drug 2: {example['Drug_B']}
        Category: {example['Category']}
        [/INST]
        Interaction: {example['Level']}
        </s>'''
    }


train_data = train_data.map(format_instruction, remove_columns=train_data.column_names)
val_data = val_data.map(format_instruction, remove_columns=val_data.column_names)


Map:   0%|          | 0/200144 [00:00<?, ? examples/s]

Map:   0%|          | 0/22239 [00:00<?, ? examples/s]

### Trying out Mistral-7B Base Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_model_id = "mistralai/Mistral-7B-v0.1"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    model_max_length=512,
    padding_side="left",
    add_eos_token=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def ask_base_model(question):
    # Use the same instruction format as your training data
    prompt = f"<s>[INST] {question} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.1, # Keep it low for medical facts
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens (the answer)
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print
    # Remove the prompt from the output to see just the answer
    answer = full_text.split("[/INST]")[-1].strip()
    return answer

In [ ]:
test_query = "How do Naltrexone and Abacavir interact?"
print(f"Question: {test_query}")
print(f"Base Model Answer: {ask_base_model(test_query)}")

Question: How do Naltrexone and Abacavir interact?
Base Model Answer: Naltrexone is a drug used to treat alcohol and opioid dependence. It is also used to treat chronic pain. Abacavir is a drug used to treat HIV. Naltrexone and Abacavir are both drugs that are used to treat different conditions. Naltrexone is a drug that is used to treat alcohol and opioid dependence. Abacavir is a drug that is used to treat HIV. Naltrexone and Abacavir are both drugs that are used to treat different conditions. Naltrexone is a drug that is


### Finetune it !!!!

configuring LORA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import EarlyStoppingCallback
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

model = prepare_model_for_kbit_training(base_model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

In [ ]:
training_arguments = SFTConfig(
    output_dir="/content/drive/MyDrive/Intelligent-Agents-Project/Models_latesttttt",
    eval_strategy="steps",      # Check performance every X steps
    eval_steps=500,                    # Frequency of evaluation
    save_steps=500,
    logging_steps=10,
    learning_rate=2e-4,
    num_train_epochs=1,
    dataset_text_field="text",
    max_length=128,
    load_best_model_at_end=True,      # Automatically keep the best version
    metric_for_best_model="loss",
    save_total_limit=2,               # Save space by only keeping recent checkpoints
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    bf16=True,
    fp16= False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,         # Use the split data
    eval_dataset=val_data,            # Use the split data
    processing_class=tokenizer,
    args=training_arguments,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Stop if no improvement for 3 checks
)

Adding EOS to train dataset:   0%|          | 0/200144 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200144 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200144 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/22239 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/22239 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/22239 [00:00<?, ? examples/s]

In [ ]:
trainer.train()
trainer.model.save_pretrained("./mistral-ddi-lora")
tokenizer.save_pretrained("./mistral-ddi-lora")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
500,0.187482,0.192686
1000,0.182483,0.183876
1500,0.181081,0.180598
2000,0.177142,0.176131
2500,0.172151,0.172975
3000,0.173143,0.171356
3500,0.168389,0.169582
4000,0.166170,0.168305
4500,0.164489,0.168064
5000,0.166996,0.165662


Step,Training Loss,Validation Loss
500,0.187482,0.192686
1000,0.182483,0.183876
1500,0.181081,0.180598
2000,0.177142,0.176131
2500,0.172151,0.172975
3000,0.173143,0.171356
3500,0.168389,0.169582
4000,0.166170,0.168305
4500,0.164489,0.168064
5000,0.166996,0.165662


In [ ]:
training_arguments = SFTConfig(
    output_dir="/content/drive/MyDrive/Intelligent-Agents-Project/Models_new",
    eval_strategy="steps",      # Check performance every X steps
    eval_steps=500,                    # Frequency of evaluation
    save_steps=500,
    logging_steps=10,
    learning_rate=2e-4,
    num_train_epochs=1,
    dataset_text_field="text",
    max_length=128,
    load_best_model_at_end=True,      # Automatically keep the best version
    metric_for_best_model="loss",
    save_total_limit=2,               # Save space by only keeping recent checkpoints
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    bf16=True,
    fp16= False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,         # Use the split data
    eval_dataset=val_data,            # Use the split data
    processing_class=tokenizer,
    args=training_arguments,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Stop if no improvement for 3 checks
)

Tokenizing train dataset:   0%|          | 0/200144 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200144 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/22239 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/22239 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/22239 [00:00<?, ? examples/s]

In [ ]:
trainer.train()
trainer.model.save_pretrained("./mistral-ddi-lora")
tokenizer.save_pretrained("./mistral-ddi-lora")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
500,0.204490,0.209599
1000,0.199592,0.200329
1500,0.206786,0.202524
2000,0.192184,0.191336
2500,0.185941,0.188102
3000,0.188493,0.186584
3500,0.182802,0.184374
4000,0.179834,0.183552
4500,0.180315,0.182596
5000,0.182660,0.180487


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel


# 1. Update this to your LATEST checkpoint folder in Drive
checkpoint_path = "/content/drive/MyDrive/Intelligent-Agents-Project/Models_new/checkpoint-7500"
model_id = "mistralai/Mistral-7B-v0.1"

# 2. BitsAndBytes Config (Must match your training settings)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. Load Base Model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Load Tokenizer & Adapters
# Note: The tokenizer should be in the checkpoint folder or the base model ID
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = PeftModel.from_pretrained(base_model, checkpoint_path)
model.eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

In [ ]:
def ask_drug_expert(question):
    prompt = f"<s>[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("[/INST]")[-1].strip()

# Example Test
print(ask_drug_expert("What is Hallucination?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Hallucination is defined as a false perception of a sensory experience that is not based on a real stimulus. Hallucinations are classified as either perceptual or cognitive. Perceptual hallucinations are characterized by the presence of sensory experiences that are not based on a real stimulus. Cognitive hallucinations are characterized by the presence of sensory experiences that are not based on a real stimulus. Hallucinations are classified as either visual, auditory, tactile, gustatory, or olfactory. Visual hallucinations are characterized by the presence of visual experiences that are not based on a real stimulus. Auditory hallucinations are characterized by the presence of auditory experiences that are not based on a real stimulus. Tactile hallucinations are characterized by the presence of tactile experiences that are not based on a real stimulus. Gustatory hallucinations are characterized by the presence of gustatory experiences that are not based on a real stimulus. Olfactory h

In [ ]:
train_data['text']

Column(['<s>[INST] How do Voriconazole and Fludrocortisone interact? [/INST] Voriconazole and Fludrocortisone are reported to have a moderate interaction. This interaction is classified under interaction involving systemic hormonal preparations, excluding sex hormones and insulins drugs. This combination may require dose adjustments or additional monitoring to ensure patient safety.</s>', '<s>[INST] How do Metolazone and Sevelamer interact? [/INST] Metolazone and Sevelamer are reported to have a unknown interaction. This interaction is classified under interaction involving various drugs. There is limited or insufficient evidence about the clinical significance of this interaction.</s>', '<s>[INST] How do Prednisolone and Liraglutide interact? [/INST] Prednisolone and Liraglutide are reported to have a moderate interaction. This interaction is classified under interaction involving dermatologicals drugs. This combination may require dose adjustments or additional monitoring to ensure p

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel


# 1. Update this to your LATEST checkpoint folder in Drive
checkpoint_path = "/content/drive/MyDrive/Intelligent-Agents-Project/Models_latesttttt/checkpoint-7500"
model_id = "mistralai/Mistral-7B-v0.1"

# 2. BitsAndBytes Config (Must match your training settings)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. Load Base Model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Load Tokenizer & Adapters
# Note: The tokenizer should be in the checkpoint folder or the base model ID
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = PeftModel.from_pretrained(base_model, checkpoint_path)
model.eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

In [ ]:
def ask_drug_expert(question):
    prompt = f"<s>[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("[/INST]")[-1].strip()

# Example Test
print(ask_drug_expert("What is hallucination?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Interaction: Moderate
